# SMA Angle Regime Classification on SPY
## Strategy Brief
This strategy involves classifying the market regime based on the angle of the Simple Moving Average (SMA). The angle of the SMA is used as a proxy for the market trend direction and strength. By classifying the market into different regimes, we can make informed decisions on whether to enter or exit trades. The strategy aims to outperform a simple buy-and-hold strategy by entering trades during favorable market conditions and exiting during unfavorable ones.
## References
- (No external references)

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## PHASE 1 - Trading Context
In this phase, we define the parameters for our trading strategy, including the lookback period for the SMA and the threshold for determining the angle.

In [ ]:
SMA_PERIOD = 50
ANGLE_THRESHOLD = 0.5
START_DATE = '2010-01-01'
END_DATE = '2023-10-01'

## PHASE 2 - Data Exploration
We will download historical price data for SPY using yfinance, calculate the Simple Moving Average (SMA), and plot it overlaid on the price data.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Download data
data = yf.download('SPY', start=START_DATE, end=END_DATE)

# Calculate SMA
data['SMA'] = data['Close'].rolling(window=SMA_PERIOD).mean()

# Plot
data[['Close', 'SMA']].plot(figsize=(14, 7))
plt.title('SPY Price and SMA')
plt.show()

## PHASE 3 - Strategy Engineering
We will calculate the angle of the SMA and classify the market regime based on the angle. A positive angle indicates an uptrend, while a negative angle indicates a downtrend.

In [ ]:
from scipy.signal import savgol_filter

# Calculate SMA angle
data['SMA_angle'] = np.rad2deg(np.arctan(savgol_filter(data['SMA'], window_length=11, polyorder=2, deriv=1)))

# Signal: 1 for uptrend, -1 for downtrend
data['Signal'] = np.where(data['SMA_angle'] > ANGLE_THRESHOLD, 1, np.where(data['SMA_angle'] < -ANGLE_THRESHOLD, -1, 0))

## PHASE 4 - Coding & Backtesting
We will implement the trading logic based on the signal, calculate daily returns, and plot the equity curve.

In [ ]:
# Positions based on signal
data['Position'] = data['Signal'].shift(1)

# Calculate daily returns
data['Market_Returns'] = data['Close'].pct_change()
data['Strategy_Returns'] = data['Position'] * data['Market_Returns']

# Calculate equity curve
data['Equity_Curve'] = (1 + data['Strategy_Returns']).cumprod()

# Plot equity curve
data['Equity_Curve'].plot(figsize=(14, 7))
plt.title('Equity Curve')
plt.show()

## PHASE 5 - Performance Evaluation
We will evaluate the performance of the strategy using metrics such as CAGR, Sharpe ratio, Sortino ratio, Calmar ratio, and maximum drawdown, and compare it to a buy-and-hold strategy.

In [ ]:
def calculate_performance(data):
    # Calculate CAGR
    total_return = data['Equity_Curve'].iloc[-1]
    n_years = (data.index[-1] - data.index[0]).days / 365.25
    cagr = (total_return) ** (1 / n_years) - 1
    
    # Calculate Sharpe ratio
    sharpe_ratio = data['Strategy_Returns'].mean() / data['Strategy_Returns'].std() * np.sqrt(252)
    
    # Calculate Sortino ratio
    downside_std = data.loc[data['Strategy_Returns'] < 0, 'Strategy_Returns'].std()
    sortino_ratio = data['Strategy_Returns'].mean() / downside_std * np.sqrt(252)
    
    # Calculate Calmar ratio
    max_drawdown = (data['Equity_Curve'].cummax() - data['Equity_Curve']).max()
    calmar_ratio = cagr / max_drawdown
    
    # Buy and hold performance
    bh_total_return = data['Close'].iloc[-1] / data['Close'].iloc[0]
    bh_cagr = (bh_total_return) ** (1 / n_years) - 1
    
    # Print results
    print(f"CAGR: {cagr:.2%}")
    print(f"Sharpe Ratio: {sharpe_ratio:.2f}")
    print(f"Sortino Ratio: {sortino_ratio:.2f}")
    print(f"Calmar Ratio: {calmar_ratio:.2f}")
    print(f"Max Drawdown: {max_drawdown:.2%}")
    print(f"Buy and Hold CAGR: {bh_cagr:.2%}")

calculate_performance(data)

## PHASE 6 - Deploy & Monitor
We will create a function to download the last 60 days of data, compute today's signal, and print the current position.

In [ ]:
def get_current_position():
    # Download last 60 days of data
    recent_data = yf.download('SPY', period='60d')
    
    # Calculate SMA
    recent_data['SMA'] = recent_data['Close'].rolling(window=SMA_PERIOD).mean()
    
    # Calculate SMA angle
    recent_data['SMA_angle'] = np.rad2deg(np.arctan(savgol_filter(recent_data['SMA'], window_length=11, polyorder=2, deriv=1)))
    
    # Determine current signal
    current_signal = np.where(recent_data['SMA_angle'].iloc[-1] > ANGLE_THRESHOLD, 1, np.where(recent_data['SMA_angle'].iloc[-1] < -ANGLE_THRESHOLD, -1, 0))
    
    # Print current position
    print(f"Current Position: {'Long' if current_signal == 1 else 'Short' if current_signal == -1 else 'Neutral'}")

get_current_position()